# PYNQ-Z2 Live Gesture Recognition & Systolic Array Accelerator

### System Architecture:
- **Heterogeneous SoC**: Xilinx Zynq-7020 (`xc7z020clg400-1`)
- **Hardware Accelerator**: Option 2 Unified Dual-Engine Systolic GEMM Accelerator
- **Compute Capacity**: **512 Physical MACs**, **220 / 220 DSPs (100% device capacity)**
- **Host Software**: ARM Cortex-A9 Linux UDP Gesture Server
- **Laptop Interface**: Real-Time MediaPipe 3D Hand Tracker with Cyberpunk HUD

### Gesture & Hardware Action Map:
- **Palm Open** &rarr; RGB LED LD4 = **GREEN** | 16×16×16 Baseline Tile (512 MACs Active)
- **Fist** &rarr; RGB LED LD4 = **RED** | 4×4×4 Power-Save Region
- **Thumbs Up** &rarr; User LED LD1 = **ON (Latched)** | 32×32×32 Multi-Tile Burst
- **Thumbs Down** &rarr; User LED LD1 = **OFF (Latched)** | 16×16×16 Tile + Soft Reset
- **Peace Sign** &rarr; 8×16×8 Partial Tile
- **Pointing** &rarr; 1×128×2 Linear FC Layer Inference
- **OK Sign** &rarr; 64×9×16 Conv1 im2col Spatial Patch

## 1. Load FPGA Accelerator Overlay & Hardware Self-Test

In [ ]:
import os, sys, time
import numpy as np
os.environ.setdefault('BOARD', 'Pynq-Z2')
os.environ.setdefault('XILINX_XRT', '/usr')

# Ensure local driver path is registered
for p in ['.', './pynq', '../pynq', '/home/xilinx/jupyter_notebooks/adaptive_gemm/pynq']:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

from pynq import Overlay
from adaptive_gemm import AdaptiveDualGemmOverlay

# Locate bitstream
bit_paths = [
    'results/adaptive_gemm.bit',
    'adaptive_gemm.bit',
    '/home/xilinx/jupyter_notebooks/adaptive_gemm/results/adaptive_gemm.bit',
    '/home/xilinx/adaptive_gemm/results/adaptive_gemm.bit'
]
bit_file = next((b for b in bit_paths if os.path.isfile(b)), None)
assert bit_file is not None, f"Could not find adaptive_gemm.bit in {bit_paths}"

print(f"[1/2] Programming FPGA bitstream: {bit_file}...")
accel = AdaptiveDualGemmOverlay(bit_file)
print(f"✅ Hardware Overlay Loaded! IP Cores: {list(accel.ol.ip_dict.keys())}")

# Run quick verification self-test
print("[2/2] Running 512-MAC Dual-Engine Self-Test against NumPy...")
np.random.seed(42)
A0 = np.random.randint(-128, 127, (16, 16), dtype=np.int8)
B0 = np.random.randint(-128, 127, (16, 16), dtype=np.int8)
A1 = np.random.randint(-128, 127, (16, 16), dtype=np.int8)
B1 = np.random.randint(-128, 127, (16, 16), dtype=np.int8)

C0_ref = np.dot(A0.astype(np.int32), B0.astype(np.int32))
C1_ref = np.dot(A1.astype(np.int32), B1.astype(np.int32))

t0 = time.perf_counter()
C0_hw, C1_hw = accel.execute_dual_tile(A0, B0, A1, B1)
dt = (time.perf_counter() - t0) * 1000
perf = accel.get_hardware_counters()

assert np.array_equal(C0_hw, C0_ref), "Engine 0 Mismatch!"
assert np.array_equal(C1_hw, C1_ref), "Engine 1 Mismatch!"
print(f"🎉 SUCCESS: Dual-Engine GEMM is 100% BIT-EXACT! Latency: {dt:.2f} ms | Cycles: {perf['cycles']}")

## 2. Launch Wireless Gesture UDP Server & Live Hardware Accelerator

In [ ]:
# Start the gesture server directly in this notebook
from pynq_gesture_server import GestureServer

print("Starting Gesture Server on UDP port 5005...")
print("On your laptop, run: START_HAND_GESTURE.bat")
print("Press Stop (■) in Jupyter to stop the server.")

server = GestureServer(listen_ip="0.0.0.0", overlay_path=bit_file)
server.start()